# Olist Data Warehouse: Building a Star Schema with DuckDB

This notebook continues `01_explore_source_data.ipynb`. The five source tables are already in the `raw` schema. Today we design a **star schema** and build it in a new schema called `dw` (data warehouse).

### What you will learn
- How to turn an operational (OLTP) data model into an analytical (OLAP) **star schema**
- What **facts**, **dimensions**, **grain** and **surrogate keys** are
- How to **validate** a warehouse (grain, orphans, reconciliation)
- How to answer business questions with OLAP operations (roll-up, slice, dice, drill-down)

### What we will do
1. Reconnect to the database and check the `raw` tables
2. Turn the ERD and the data quality findings into design decisions
3. Create the `dw` schema
4. Build five dimension tables
5. Build two fact tables
6. Validate the warehouse: keys, orphans, reconciliation
7. Declare primary keys

> **How to work with this notebook:** run the cells from top to bottom. Read the text above each code cell *before* you run it, and look at the output *after* you run it. Every table is created with `CREATE OR REPLACE`, so you can safely re-run any cell.

### The big picture: what a star schema looks like

A star schema has a **fact table** in the middle (the events we measure) surrounded by **dimension tables** (the context: who, what, where, when). Drawn on paper it looks like a star.

```
                         dim_date
                            |
   dim_customer ---- fact_order_items ---- dim_product
                            |
                       dim_seller
```

We build **two** fact tables, because the business asks questions at two different levels of detail:

- `fact_order_items`: one row per product line in an order (good for revenue by product, seller, category)
- `fact_orders`: one row per order (good for delivery performance and order status)

Both share some of the same dimensions (`dim_customer`, `dim_date`). Sharing dimensions between fact tables is called **conformed dimensions**.

## 1. Setup and reconnect

### 1.1 Imports

- `pathlib.Path`: builds file paths that work on any operating system
- `duckdb`: the database engine. DuckDB runs inside Python and stores everything in a single file
- `pandas`: holds query results as tables (DataFrames) that Jupyter displays nicely
- `matplotlib`: draws charts

In [7]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

### 1.2 Connect to the database

The database file from last session holds the `raw` tables. We open **the same file**, so everything we built then is still there.

- `Path.cwd().parent` is the folder above the folder this notebook is in (the project root)
- The `assert` line stops the notebook with a clear message if the file is missing. That is better than a confusing error later
- `duckdb.connect(...)` opens the file. The variable `con` is our *connection*: every SQL command we send goes through it

In [8]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
WAREHOUSE_DIR = DATA_DIR / "warehouse"
DB_PATH = WAREHOUSE_DIR / "ecommerce.duckdb"

RAW_SCHEMA = "raw"
DW_SCHEMA = "dw"
RAW_TABLES = ["customers", "orders", "order_items", "products", "sellers"]

assert DB_PATH.exists(), f"{DB_PATH} not found. Run 01_explore_source_data.ipynb first."
con = duckdb.connect(str(DB_PATH))

### 1.3 Helper functions

We will run many queries and many checks, so we write a few small helper functions once and reuse them. Each one wraps a DuckDB call in a single, readable name.

| Helper | What it does | Example use |
|---|---|---|
| `q(sql)` | runs a query and returns a **table** (a pandas DataFrame). Jupyter shows it nicely when it is the last line of a cell | `q("SELECT * FROM dw.dim_seller")` |
| `run(sql)` | runs a statement that **builds or changes** something (`CREATE`, `ALTER`...). Nothing is displayed | `run("CREATE SCHEMA ...")` |
| `scalar(sql)` | runs a query that returns **one value** and gives you that value | `scalar("SELECT count(*) FROM raw.orders")` |
| `row(sql)` | runs a query that returns **one row** and gives you that row as a tuple | `row("SELECT min(x), max(x) FROM t")` |
| `check(name, actual, expected)` | compares two numbers and prints **PASS** or **FAIL**. On FAIL it stops the notebook | see section 4 onwards |

`check` is our automatic safety net: *if the warehouse is wrong, we want to know immediately, not three steps later*.

In [ ]:
def q(sql_str: str) -> pd.DataFrame:
    """Run a SQL query in DuckDB and return a pandas DataFrame."""
    return con.sql(sql_str).df()


def run(sql_str: str) -> None:
    """Run a SQL statement that builds or changes something (CREATE, ALTER...)."""
    con.execute(sql_str)


def row(sql_str: str) -> tuple:
    """Run a query and return its first row as a tuple."""
    return con.execute(sql_str).fetchall()[0]


def scalar(sql_str: str):
    """Run a query that returns a single value and return that value."""
    return row(sql_str)[0]


def check(name: str, actual, expected) -> None:
    """Print PASS or FAIL for one validation, and stop the notebook on FAIL."""
    ok = actual == expected
    print(f"{'PASS' if ok else 'FAIL'}  {name}: got {actual:,}, expected {expected:,}")
    assert ok, f"{name} failed"

### 1.4 Check that the raw tables from last session are there

We count the rows of each raw table and keep the counts in a dictionary called `raw_counts`. Later we compare the warehouse against these numbers: *nothing should be lost or duplicated on the way from `raw` to `dw`*.

In [ ]:
raw_counts = {t: scalar(f"SELECT count(*) FROM {RAW_SCHEMA}.{t}") for t in RAW_TABLES}
pd.DataFrame(raw_counts.items(), columns=["table_name", "row_count"])

## 2. From the ERD to a star schema

### 2.1 OLTP versus OLAP

The `raw` tables come from an **operational system** (OLTP). A warehouse is built for a different job (OLAP).

| | Source system (OLTP) | Data warehouse (OLAP) |
|---|---|---|
| Purpose | Run the business: take orders | Analyse the business: answer questions |
| Design | Many small normalised tables | Few wide tables around a fact (star schema) |
| Typical query | One order by its id | Revenue by state and quarter |
| Joins per query | Many | Few |

### 2.2 The four design steps (Kimball method)

Always design in this order:

1. **Choose the business process:** the sale of products on the marketplace.
2. **Declare the grain:** exactly what **one row** of the fact table represents. This is the most important decision: everything else follows from it.
3. **Identify the dimensions:** the descriptive context (who, what, where, when).
4. **Identify the facts:** the numeric measures at that grain.

### 2.3 Our design

| Fact table | Grain (one row per...) | Measures |
|---|---|---|
| `fact_order_items` | order line | `price`, `freight_value`, `line_total` |
| `fact_orders` | order | `n_items`, `items_value`, `freight_value`, `delivery_days`, `is_late` |

| Dimension | Grain (one row per...) | Business key | Type |
|---|---|---|---|
| `dim_customer` | real person | `customer_unique_id` | one row per key |
| `dim_product` | product | `product_id` | one row per key |
| `dim_seller` | seller | `seller_id` | one row per key |
| `dim_order_status` | status value | `order_status` | one row per key |
| `dim_date` | calendar day | `date_key` (YYYYMMDD) | generated |

### 2.4 Design decisions from the data quality findings

Last session we found problems in the source data. Each one becomes a design decision:

| Finding in class 1 | Design decision |
|---|---|
| `customer_id` is created **per order**, so one real person has several `customer_id`s | `dim_customer` is built around `customer_unique_id` (the real person) |
| A person's address can differ between their records | We keep the address from the person's **latest order** |
| Many products are sold by several sellers | `dim_seller` is separate; the fact table holds `seller_key` |
| 775 orders have no items | They stay in `fact_orders`, which is at order grain |
| Products without category | Replace with `'unknown'` |
| Undelivered orders have no delivery date | The delivery date key is left null |

### 2.5 Surrogate keys vs business keys

- The **business key** comes from the source system (`product_id`, `customer_unique_id`). It is a long text string and the source system owns it
- The **surrogate key** is a key that the warehouse generates itself. It is independent of the source system. It can be a simple integer (`product_key = 1, 2, 3...`, small and fast) or a **hash** of the business key (more stable, see `dim_customer`)

Every dimension gets a surrogate key. The business key stays as a normal column, so while loading the facts we can look up *business key → surrogate key*.

**Why a hash for `dim_customer`?** A counter such as `row_number()` is **not stable**: the number a customer gets depends on the order and the amount of data in this particular load. After a reload, or when new customers arrive, the same customer can get a *different* number, and every fact row that stored the old number now points to the wrong customer. A hash such as `md5(customer_unique_id)` is **deterministic**: the same input always gives the same key, on every run and every machine. It needs no counter, and keys never change between loads. The price is that the key is a 32-character text (`VARCHAR`) instead of a small integer.

## 3. Create the `dw` schema

A **schema** is a folder inside the database that groups tables. We keep the warehouse in its own schema (`dw`), separate from the untouched source data in `raw`. That way we can always rebuild the warehouse from `raw`.

In [ ]:
run(f"CREATE SCHEMA IF NOT EXISTS {DW_SCHEMA}")

## 4. Build the dimensions

We build the dimensions **before** the facts, because a fact row needs the surrogate keys of its dimensions.

### 4.1 `dim_customer`

#### Step A: understand the problem in the raw data

In `raw.customers`, every **order** gets a fresh `customer_id`. The *real person* is identified by `customer_unique_id`. So one person can appear in several rows. Here is one person who bought from **different states**:

In [ ]:
example_id = scalar("""
    SELECT customer_unique_id
    FROM raw.customers
    GROUP BY customer_unique_id
    HAVING count(DISTINCT customer_state) > 1
    ORDER BY customer_unique_id
    LIMIT 1
""")

q(f"""
SELECT c.customer_id, c.customer_unique_id,
       c.customer_city, c.customer_state,
       o.order_purchase_timestamp
FROM raw.customers c
JOIN raw.orders o USING (customer_id)
WHERE c.customer_unique_id = '{example_id}'
ORDER BY o.order_purchase_timestamp
""")

Notice: one `customer_unique_id` (one person), several `customer_id`s (one per order), and the **address changes** between orders.

#### Step B: how often does this happen?

For each person we count how many records they have and how many *different* zip codes, cities and states appear. Read the query from the inside out: the inner query counts per person, the outer query counts how many people have more than one.

In [ ]:
q("""
WITH per_person AS (          -- one row per real person
    SELECT customer_unique_id,
           count(*)                                 AS n_ids,
           count(DISTINCT customer_zip_code_prefix) AS n_zip,
           count(DISTINCT customer_city)            AS n_city,
           count(DISTINCT customer_state)           AS n_state
    FROM raw.customers
    GROUP BY customer_unique_id
)
SELECT count(*)                            AS people,
       count(*) FILTER (WHERE n_ids > 1)   AS with_several_records,
       count(*) FILTER (WHERE n_zip > 1)   AS different_zip,
       count(*) FILTER (WHERE n_city > 1)  AS different_city,
       count(*) FILTER (WHERE n_state > 1) AS different_state
FROM per_person
""")

It is a small share of all people, but not zero. We need a simple rule so that each person becomes **exactly one row**. **Rule: use the address from the person's latest order.**

#### Step C: build the table

The query has three ideas:

1. **Join `raw.customers` to `raw.orders`**, so every customer record gets the date of its order.
2. **Rank each person's records from newest to oldest.** `row_number() OVER (PARTITION BY customer_unique_id ORDER BY order_purchase_timestamp DESC)` numbers the records of each person: the latest order gets `1`. We add `order_id` to the `ORDER BY` as a tie-breaker, so the result is always the same.
3. **Keep only rank 1.** `QUALIFY` filters on the result of a window function, so we do not need a subquery. (`QUALIFY` is to window functions what `HAVING` is to `GROUP BY`.)

The surrogate key is `md5(customer_unique_id)`. One row per person means the business key alone identifies the row, so we hash just that.

In [ ]:
run("""
CREATE OR REPLACE TABLE dw.dim_customer AS
SELECT md5(c.customer_unique_id)  AS customer_key,
       c.customer_unique_id,
       c.customer_zip_code_prefix AS zip_code_prefix,
       c.customer_city            AS city,
       c.customer_state           AS state
FROM raw.customers c
JOIN raw.orders o USING (customer_id)
QUALIFY row_number() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id
        ) = 1
""")

#### Step D: check the result

We must have **exactly one row per person**: the number of rows must equal the number of distinct `customer_unique_id` in the raw data.

In [ ]:
check("dim_customer rows = distinct customer_unique_id in raw",
      scalar("SELECT count(*) FROM dw.dim_customer"),
      scalar("SELECT count(DISTINCT customer_unique_id) FROM raw.customers"))
q("SELECT * FROM dw.dim_customer LIMIT 5")

Now the same person as in Step A. In the raw data they had several records. In the dimension they have **one row**, with the address of their latest order:

In [ ]:
q(f"""
SELECT customer_key, customer_unique_id, city, state
FROM dw.dim_customer
WHERE customer_unique_id = '{example_id}'
""")

> **A simplification to be aware of:** because the dimension keeps only the latest address, a person who moved appears in their *latest* state for **all** their orders, including the old ones.

### 4.2 `dim_seller`

A straight copy of `raw.sellers` with a surrogate key and cleaner column names. We drop the `seller_` prefix because the table name already says it is about sellers. This dimension is **type 1**: one row per seller, no history.

`row_number() OVER (ORDER BY seller_id)` hands out the numbers 1, 2, 3... which become the surrogate key.

In [ ]:
run("""
CREATE OR REPLACE TABLE dw.dim_seller AS
SELECT md5(seller_id) AS seller_key,
       seller_id,
       seller_zip_code_prefix AS zip_code_prefix,
       seller_city            AS city,
       seller_state           AS state
FROM raw.sellers
""")

check("dim_seller rows = raw.sellers rows",
      scalar("SELECT count(*) FROM dw.dim_seller"),
      raw_counts["sellers"])
q("SELECT * FROM dw.dim_seller LIMIT 5")

### 4.3 `dim_product`

Cleaning rules applied here:
- Missing category becomes `'unknown'` (with `coalesce`, which returns the first non-null value), so a join or group by never silently drops those products.
- The three size columns are combined into one derived attribute, `volume_cm3`.
- The misspelled raw columns (`product_name_lenght`) are left out. We only keep columns that are useful for analysis.

In [ ]:
run("""
CREATE OR REPLACE TABLE dw.dim_product AS
SELECT md5(product_id) AS product_key,
       product_id,
       coalesce(product_category_name, 'unknown')               AS category,
       product_photos_qty                                       AS photos_qty,
       product_weight_g                                         AS weight_g,
       product_length_cm                                        AS length_cm,
       product_height_cm                                        AS height_cm,
       product_width_cm                                         AS width_cm,
       product_length_cm * product_height_cm * product_width_cm AS volume_cm3
FROM raw.products
""")

check("dim_product rows = raw.products rows",
      scalar("SELECT count(*) FROM dw.dim_product"),
      raw_counts["products"])
check("products with unknown category",
      scalar("SELECT count(*) FROM dw.dim_product WHERE category = 'unknown'"),
      scalar("SELECT count(*) FROM raw.products WHERE product_category_name IS NULL"))
q("SELECT * FROM dw.dim_product LIMIT 5")

### 4.4 `dim_order_status`

A very small dimension with the distinct status values (`delivered`, `shipped`, `canceled`...). It lets us group and filter on status without storing text in every fact row. The fact table stores the small integer `status_key` instead.

In [ ]:
run("""
CREATE OR REPLACE TABLE dw.dim_order_status AS
SELECT md5(order_status) AS status_key,
       order_status
FROM (SELECT DISTINCT order_status FROM raw.orders)
""")
q("SELECT * FROM dw.dim_order_status")

### 4.5 `dim_date`

A date dimension is **generated**, not loaded from the source. It has one row for every calendar day and pre-computes everything we usually group by: year, quarter, month, weekday, weekend flag. This means analysts never need to write date functions in their queries.

It must cover **every date used by the facts**: from the first purchase to the last *estimated* delivery date.

How the query works:
- `generate_series(start, end, INTERVAL 1 DAY)` creates a list with every day between the two dates
- `unnest(...)` turns that list into one row per day
- The outer `SELECT` derives the descriptive columns from each date `d`

The key is an integer in the form `YYYYMMDD` (for example `20180131`). It is readable, and it sorts correctly.

In [ ]:
run("""
CREATE OR REPLACE TABLE dw.dim_date AS
SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER) AS date_key,
       CAST(d AS DATE)                        AS full_date,
       year(d)                                AS calendar_year,
       quarter(d)                             AS calendar_quarter,
       month(d)                               AS calendar_month,
       strftime(d, '%B')                      AS month_name,
       dayofweek(d)                           AS day_of_week,
       strftime(d, '%A')                      AS day_name,
       dayofweek(d) IN (0, 6)                 AS is_weekend
FROM (
    SELECT unnest(generate_series(
        (SELECT min(order_purchase_timestamp)::DATE      FROM raw.orders),
        (SELECT max(order_estimated_delivery_date)::DATE FROM raw.orders),
        INTERVAL 1 DAY
    )) AS d
)
""")
q("SELECT count(*) AS days, min(full_date) AS first_day, max(full_date) AS last_day FROM dw.dim_date")

## 5. Build the fact tables

A fact row stores **foreign keys to the dimensions** plus the **measures** (the numbers we add up). To fill the keys we join each dimension on its business key and take its surrogate key.

> **Grain check:** joining to dimensions must not change the number of fact rows. If the count grows, some dimension key is not unique and the facts would be double counted. If the count shrinks, some rows found no match and were lost. Either way the warehouse is wrong, so we check the row count after every build.

### 5.1 `fact_order_items`: one row per order line

Grain: **one product line in one order**. In the raw data the order line is identified by `(order_id, order_item_id)`.

| Column type | Columns |
|---|---|
| Identifiers (kept for traceability) | `order_id`, `order_item_id` |
| Foreign keys | `customer_key`, `product_key`, `seller_key`, `purchase_date_key` |
| Measures | `price`, `freight_value`, `line_total` |

`line_total` is a *derived measure* (price + freight) computed once while loading.

To find the `customer_key` of an order line we follow the path **order line → order → `raw.customers`** (which gives `customer_unique_id`) **→ `dim_customer`**.

In [ ]:
run("""
CREATE OR REPLACE TABLE dw.fact_order_items AS
SELECT i.order_id,
       i.order_item_id,
       dc.customer_key,
       dp.product_key,
       ds.seller_key,
       CAST(strftime(o.order_purchase_timestamp, '%Y%m%d') AS INTEGER) AS purchase_date_key,
       i.price,
       i.freight_value,
       i.price + i.freight_value AS line_total
FROM raw.order_items i
JOIN raw.orders    o  USING (order_id)
JOIN raw.customers rc USING (customer_id)
JOIN dw.dim_customer dc ON dc.customer_unique_id = rc.customer_unique_id
JOIN dw.dim_product  dp ON dp.product_id         = i.product_id
JOIN dw.dim_seller   ds ON ds.seller_id          = i.seller_id
""")

check("fact_order_items rows = raw.order_items rows (grain preserved)",
      scalar("SELECT count(*) FROM dw.fact_order_items"),
      raw_counts["order_items"])
q("SELECT * FROM dw.fact_order_items LIMIT 5")

## 6. Validate the warehouse

Never trust a load without checking it. We run four kinds of check, all written as queries so they can be re-run after every load. Each one calls `check(...)`: if one fails, the notebook stops.

### 6.1 Business keys are unique in each dimension

In every dimension the business key must not repeat: one row per key. If it repeats, joining a fact table to the dimension would duplicate fact rows.

In [ ]:
dim_keys = {
    "dim_customer": "customer_unique_id",
    "dim_product": "product_id",
    "dim_seller": "seller_id",
    "dim_order_status": "order_status",
}
for table, business_key in dim_keys.items():
    n, n_distinct = row(f"SELECT count(*), count(DISTINCT {business_key}) FROM dw.{table}")
    check(f"{table}.{business_key} is unique", n_distinct, n)

### 6.2 No orphans: every foreign key in a fact table exists in its dimension

An **orphan** is a fact row whose foreign key points to nothing. We look for them with an `ANTI JOIN`, which returns the rows on the left that have **no match** on the right. The expected result is always `0`.

Note the last check: `delivered_date_key` may legitimately be null (not delivered yet), so we only test the rows where it is set.

In [ ]:
orphan_checks = {
    "fact_order_items -> dim_customer": "SELECT count(*) FROM dw.fact_order_items f ANTI JOIN dw.dim_customer d USING (customer_key)",
    "fact_order_items -> dim_product":  "SELECT count(*) FROM dw.fact_order_items f ANTI JOIN dw.dim_product  d USING (product_key)",
    "fact_order_items -> dim_seller":   "SELECT count(*) FROM dw.fact_order_items f ANTI JOIN dw.dim_seller   d USING (seller_key)",
    "fact_order_items -> dim_date":     "SELECT count(*) FROM dw.fact_order_items f ANTI JOIN dw.dim_date d ON f.purchase_date_key = d.date_key"
}
for name, sql in orphan_checks.items():
    check(f"orphans, {name}", scalar(sql), 0)

### 6.3 Reconciliation: the warehouse totals must match the source

The most convincing test: add up the money in the source and in the warehouse. If the totals differ, rows were lost or duplicated somewhere in the load.

In [ ]:
raw_totals = row("SELECT round(sum(price), 2), round(sum(freight_value), 2) FROM raw.order_items")
dw_totals  = row("SELECT round(sum(price), 2), round(sum(freight_value), 2) FROM dw.fact_order_items")

check("revenue: fact_order_items vs raw", dw_totals[0], raw_totals[0])
check("freight: fact_order_items vs raw", dw_totals[1], raw_totals[1])

## 7. Declare the keys

In the last session we **proved** keys with queries. A warehouse also **declares** them, so the database itself rejects bad rows. We add a primary key to every dimension and fact table.

- Dimensions: the surrogate key (`customer_key`, `product_key`...). `customer_key` is the md5 hash
- `fact_order_items`: a **composite** key of two columns, `(order_id, order_item_id)`
- `fact_orders`: `order_id`

The `try/except` prints a message instead of crashing if you re-run the cell and the key already exists.

In [ ]:
primary_keys = {
    "dim_customer": "customer_key",
    "dim_product": "product_key",
    "dim_seller": "seller_key",
    "dim_order_status": "status_key",
    "dim_date": "date_key",
    "fact_order_items": "order_id, order_item_id",
    "fact_orders": "order_id",
}
for table, cols in primary_keys.items():
    try:
        con.execute(f"ALTER TABLE dw.{table} ADD PRIMARY KEY ({cols})")
        print(f"added PRIMARY KEY ({cols}) to dw.{table}")
    except duckdb.Error as err:
        print(f"dw.{table}: {str(err).splitlines()[0]}")

The key now shows in `DESCRIBE`, and a duplicate row is rejected:

In [ ]:
display(q("DESCRIBE dw.dim_seller")[["column_name", "column_type", "null", "key"]])

try:
    con.execute("INSERT INTO dw.dim_seller SELECT * FROM dw.dim_seller LIMIT 1")
except duckdb.Error as err:
    print("Rejected:", str(err).splitlines()[0])

## 8. OLAP on the star schema

Now we use the warehouse. Compare how many tables a revenue-by-quarter query needs:

**On the raw tables:** `orders` + `order_items` + (a date expression) with no ready-made year or quarter.

**On the star schema:** the fact table + `dim_date`, which already holds year and quarter.

> **Note:** when we group by `c.state` in the queries below, it is the customer's **latest** address (see the simplification in section 4.1).

We restrict analysis to **Jan 2017 to Aug 2018**, the months with full coverage found in last session. We store the filter in a Python variable so we can reuse it in every query with an f-string (`{FULL_PERIOD}`).

In [ ]:
FULL_PERIOD = "d.full_date BETWEEN '2017-01-01' AND '2018-08-31'"

### Slice: fix one dimension (year = 2018)

A **slice** keeps only one value of one dimension (here: the year 2018) and looks at the rest. The chart shows the top 10 customer states by revenue.

In [ ]:
by_state = q(f"""
SELECT c.state, round(sum(f.price)) AS revenue
FROM dw.fact_order_items f
JOIN dw.dim_date     d ON f.purchase_date_key = d.date_key
JOIN dw.dim_customer c USING (customer_key)
WHERE d.calendar_year = 2018 AND {FULL_PERIOD}
GROUP BY c.state ORDER BY revenue DESC LIMIT 10
""")
ax = by_state.plot.barh(x="state", y="revenue", figsize=(8, 4), legend=False, color="#2a7de1")
ax.invert_yaxis(); ax.set_title("2018 revenue by customer state (top 10)"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

### Dice: restrict several dimensions at once

A **dice** selects a sub-cube: several dimensions are restricted at the same time (year, customer state and product category).

In [ ]:
q(f"""
SELECT c.state AS customer_state, p.category, round(sum(f.price)) AS revenue
FROM dw.fact_order_items f
JOIN dw.dim_date     d ON f.purchase_date_key = d.date_key
JOIN dw.dim_customer c USING (customer_key)
JOIN dw.dim_product  p USING (product_key)
WHERE d.calendar_year = 2018 AND {FULL_PERIOD}
  AND c.state IN ('SP', 'RJ')
  AND p.category IN ('beleza_saude', 'esporte_lazer')
GROUP BY 1, 2 ORDER BY 1, 3 DESC
""")

### Key ideas to remember

| Concept | In one sentence |
|---|---|
| **Star schema** | A fact table in the middle, surrounded by dimension tables |
| **Grain** | What exactly one row of a fact table means. Decide it first |
| **Fact / measure** | A number we add up (price, freight, delivery days) |
| **Dimension** | The descriptive context we group and filter by (customer, product, date...) |
| **Surrogate key** | A key generated by the warehouse (an integer, or a stable hash like `md5(...)`); the business key stays as a normal column |
| **Orphan** | A fact row whose foreign key finds no row in the dimension |
| **Reconciliation** | Comparing warehouse totals with the source to prove nothing was lost or duplicated |